<a href="https://colab.research.google.com/github/specM7/DSGP_Group_33_Brain_Tumor_Predictor/blob/NoTumor-%26-Chatbot-Ahshaan-2506751/TinyLlama_Model01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install required libraries for training and evaluation

!pip install transformers datasets accelerate evaluate sentencepiece

In [5]:
# Mount Google Drive to access dataset and save trained model

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Load brain tumor question-answer dataset from Google Drive

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/Chatbot/dataset.json"
)

dataset = dataset["train"]

print("Dataset size:", len(dataset))

Generating train split: 0 examples [00:00, ? examples/s]

Dataset size: 508


In [7]:
# System instruction for chatbot behaviour

system_prompt = """
You are a medical AI assistant specialized in brain tumor education and MRI result guidance.

You help users understand:
- Brain tumors and related conditions
- Glioma tumors
- Meningioma tumors
- Pituitary tumors
- Brain MRI scans and MRI reports
- Brain tumor symptoms and warning signs
- Diagnosis methods such as MRI, biopsy, and neurological exams
- Treatment options including surgery, radiation therapy, and chemotherapy
- Recovery, follow-up scans, and patient support

Your goal is to provide clear, supportive, and educational answers for patients who may be worried about brain tumor symptoms or MRI results.

Important rules:
- Provide simple and easy-to-understand medical explanations.
- Do not make a final medical diagnosis.
- Encourage users to consult a neurologist or doctor for professional medical advice.
- Be supportive and calm when users express fear or anxiety.

If a question is unrelated to brain tumors, MRI scans, or brain health, respond politely:

"I am designed to help with brain tumor and brain MRI related questions. Please consult the appropriate professional for other topics."
"""

In [8]:
# Load TinyLlama model and tokenizer from HuggingFace

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully


In [9]:
# Convert dataset into prompt format for LLM training

def format_data(example):

    prompt = f"""
{system_prompt}

User: {example['instruction']}

Assistant: {example['output']}
"""

    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(format_data)

print("Dataset formatted")

Map:   0%|          | 0/508 [00:00<?, ? examples/s]

Dataset formatted


In [10]:
# Define training parameters

from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="/content/brain_tumor_model",

    per_device_train_batch_size=2,

    num_train_epochs=2,   # Full fine-tuning for 2 epochs

    learning_rate=2e-5,

    logging_steps=20,

    save_steps=200,

    save_total_limit=1
)

In [11]:
# Initialize HuggingFace Trainer

from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset
)

In [12]:
# Start model fine-tuning

train_output = trainer.train()

print("Training completed")

Step,Training Loss
20,0.194864
40,0.000069
60,0.000054
80,0.000048
100,0.000045
120,0.000041
140,0.000039
160,0.000037
180,0.000036
200,0.000034


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed


In [13]:
# Display final training loss

training_loss = train_output.training_loss

print("Final Training Loss:", training_loss)

Final Training Loss: 0.007705706715039308


In [14]:
# Save trained model locally in Colab

trainer.save_model("/content/brain_tumor_chatbot")

tokenizer.save_pretrained("/content/brain_tumor_chatbot")

print("Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved


In [15]:
# Copy trained model to Google Drive

!cp -r /content/brain_tumor_chatbot /content/drive/MyDrive/Chatbot/

print("Model saved to Google Drive")

Model saved to Google Drive


In [16]:
# Evaluate chatbot using BLEU score

import evaluate

bleu = evaluate.load("bleu")

predictions = []
references = []

for sample in dataset.select(range(50)):

    prompt = f"User: {sample['instruction']}\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=60)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictions.append(response)

    references.append([sample["output"]])

score = bleu.compute(predictions=predictions, references=references)

print("BLEU Score:", score)

BLEU Score: {'bleu': 0.030416930899243636, 'precisions': [0.16293134115972963, 0.039478449837015574, 0.014754703061600885, 0.009019165727170236], 'brevity_penalty': 1.0, 'length_ratio': 2.61731843575419, 'translation_length': 2811, 'reference_length': 1074}


In [17]:
# Calculate model perplexity

import torch
import math

loss = training_loss

perplexity = math.exp(loss)

print("Perplexity:", perplexity)

Perplexity: 1.0077354720782952


In [18]:
# Simple chatbot test

from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="/content/brain_tumor_chatbot",
    tokenizer=tokenizer
)

question = "What is glioma tumor?"

prompt = f"User: {question}\nAssistant:"

result = chatbot(prompt, max_new_tokens=80)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: What is glioma tumor?
Assistant: Glioma tumors are brain tumors that form in the brain and are characterized by abnormal growth of glial cells. They can be benign (non-cancerous) or malignant (cancerous). Glioma tumors can cause symptoms such as neck pain, difficulty with movement, and loss of balance. Malignant glioma tumors are more likely to
